# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

This section prints all available record sets and their fields, referencing each by its `@id`.

In [ ]:
# List all record sets and their fields by their @id
print("Available Record Sets:")
record_sets = []
for record_set in metadata.record_sets:
    print(f"- RecordSet name: {record_set.name}, @id: {record_set.id}")
    record_sets.append(record_set.id)
    if hasattr(record_set, 'fields'):
        print("  Fields (by @id):")
        for field in record_set.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print()

if not record_sets:
    print("No record sets found — please check dataset or schema structure.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = dict()

for record_set_id in record_sets:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"  No records extracted for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For this example, we'll use the first available record set and its numeric fields
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f"Using RecordSet @id: {main_record_set_id}")

    # Identify numeric fields
    # Try to choose columns that look like numeric values
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_cols:
        # Also consider columns with 'amount', 'count', 'percent', 'MW', 'Num', 'Abundance' in their names
        for col in df.columns:
            if any(sub in col.lower() for sub in ['amount', 'count', 'percent', 'mw', 'num', 'abundance', 'coverage']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except Exception:
                    continue
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Selected numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalizing
        colnorm = f"{numeric_field}_normalized"
        filtered_df[colnorm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, colnorm]].head())

        # Try to group by a field (e.g. first non-numeric field)
        other_fields = [col for col in df.columns if col != numeric_field]
        group_field = None
        for col in other_fields:
            if df[col].dtype == object and df[col].nunique() < len(df)//2 and df[col].nunique() < 30:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
    else:
        print("No numeric fields found to analyze.")
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

# Demonstrate a histogram and boxplot for the selected numeric field
if len(dataframes) > 0 and numeric_cols:
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    df[numeric_field].hist(bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_field])
    plt.title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()

    # If a group_field was chosen, plot a bar chart
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print('No numeric fields to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides comprehensive data from mass spectrometry experiments of human mast cell vesicles.
- We demonstrated loading Croissant metadata, enumerating record sets and fields using their `@id`s, extracting records, and basic filtering/grouping.
- Exploratory steps included filtering based on numeric criteria, normalization, and visualization of distributions.

**Next steps:**
- Explore rich relationships among fields for predictive analysis or statistical modeling.
- Extend with domain-specific biological queries, e.g. filter by peptide abundance or protein IDs.
- Evaluate and clean any missing data relevant for downstream tasks.